# Memory-Native Qwen3.8-27B — Strict KD v3

Production notebook for **H100** or **RTX PRO 6000 Blackwell 96 GB**.

This v3 fixes the v2 import/path bug **and** the deeper correctness problems:

- text-only student (vision/MTP are never materialized);
- exact runner path variables (`MODEL`, `STATE_DIR`, `DATA_DIR`, `CACHE`, `CKPT_DIR`);
- teacher cache uses **top-k + full-vocabulary tail mass**, not renormalized top-k;
- cache carries the exact `input_ids` and is verified step-by-step;
- `alpha=0` for every train/eval forward; `c` is only the counter accumulator;
- `decimation=1`; no ordinary gradient accumulation (counter layers mutate on every backward);
- 400 dec1 steps ≈ the old 1500/dec4 per-group update count, with ~4x smaller LR;
- all ordinary FP weights frozen by default; only counter state/scale can move;
- warm step 0 is protected; every KD eval is gated *before* selection;
- `best.pt` is written only for a deployable candidate that passes all domain gates;
- automatic early stop if strict PPL never beats the warm state.

**Run every cell in order. Do not reuse a v2 teacher cache.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# ---- Canonical Drive paths ----
from pathlib import Path
import os, shutil, json, subprocess, sys, zipfile

RELEASE_ZIP = Path('/content/drive/MyDrive/mn_strict_kd_qwen38_27b_h100_rtxpro6000_v3.zip')
DONOR_DIR = Path('/content/drive/MyDrive/colab_models/Qwen3.8-27B')
WARM_STATE_ZIP = Path('/content/drive/MyDrive/mn27/state_max.zip')
DATA_DRIVE = Path('/content/drive/MyDrive/mn27/mix_qwen38_v3')
OUTPUT_DRIVE = Path('/content/drive/MyDrive/mn27_strict_v3')
CACHE_DRIVE = OUTPUT_DRIVE / 'teacher_cache_v3'

WORK = Path('/content/mn_strict_v3')
PROJECT = WORK / 'project'
LOCAL_OUTPUT = WORK / 'output'
LOCAL_CACHE = WORK / 'teacher_cache_v3'
LOCAL_DATA = WORK / 'mix_qwen38_v3'
TMP_CKPT = WORK / 'ckpt_tmp'
for p in (WORK, LOCAL_OUTPUT, TMP_CKPT): p.mkdir(parents=True, exist_ok=True)

for p in (RELEASE_ZIP, DONOR_DIR, WARM_STATE_ZIP):
    assert p.exists(), f'Missing: {p}'
print('Paths OK')

In [ ]:
# Hardware + storage preflight
import torch, shutil, os
assert torch.cuda.is_available(), 'CUDA GPU required'
props = torch.cuda.get_device_properties(0)
name = props.name
gib = props.total_memory / 2**30
print(f'GPU: {name} | {gib:.2f} GiB | cc={torch.cuda.get_device_capability(0)} | bf16={torch.cuda.is_bf16_supported()}')
assert gib >= 72, '27B strict recipe requires >=72 GiB VRAM'
assert torch.cuda.is_bf16_supported(), 'BF16 support required'
local_free = shutil.disk_usage('/content').free / 2**30
shm_free = shutil.disk_usage('/dev/shm').free / 2**30
print(f'free /content={local_free:.1f} GiB | /dev/shm={shm_free:.1f} GiB')
assert local_free >= 35, 'Need >=35 GiB free local disk even when donor stays on Drive'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:256'
os.environ['TOKENIZERS_PARALLELISM'] = 'false' 

In [ ]:
# Extract v3 and install without touching Colab's CUDA-matched torch.
import shutil, zipfile, subprocess, sys, importlib
if PROJECT.exists(): shutil.rmtree(PROJECT)
tmp = WORK / 'extract_tmp'
if tmp.exists(): shutil.rmtree(tmp)
tmp.mkdir(parents=True)
with zipfile.ZipFile(RELEASE_ZIP) as zf: zf.extractall(tmp)
roots = [p for p in tmp.iterdir() if p.is_dir()]
src_root = roots[0] if len(roots) == 1 else tmp
shutil.move(str(src_root), str(PROJECT))

# src-layout: add PROJECT/src explicitly (this is the v2 import bug fix).
SRC = PROJECT / 'src'
assert (SRC / 'mn_strict_kd' / 'hardware.py').exists(), SRC
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
importlib.invalidate_caches()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT/'requirements-colab.txt')], check=True)
from mn_strict_kd.hardware import detect
profile = detect()
print(profile)

In [ ]:
# Ensure Transformers knows Qwen3.8, but never let pip replace torch/CUDA.
import importlib, subprocess, sys
ok = False
try:
    import transformers
    ok = hasattr(transformers, 'Qwen3_8ForCausalLM') and hasattr(transformers, 'Qwen3_8ForConditionalGeneration')
except Exception:
    pass
if not ok:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', '--no-deps', 'git+https://github.com/huggingface/transformers.git'], check=True)
    importlib.invalidate_caches()
    import transformers
print('transformers', transformers.__version__, 'Qwen3.8=', hasattr(transformers, 'Qwen3_8ForCausalLM'))
assert hasattr(transformers, 'Qwen3_8ForCausalLM')

In [ ]:
# Static release/donor/warm-state validation.
import subprocess, sys, json
preflight = LOCAL_OUTPUT / 'preflight_v3.json'
subprocess.run([
    sys.executable, str(PROJECT/'scripts/validate_strict_release_v3.py'),
    '--donor', str(DONOR_DIR), '--state', str(WARM_STATE_ZIP),
    '--output', str(preflight), '--require-gpu'
], check=True)
print(preflight.read_text())

In [ ]:
# GPU/Triton counter smoke on the ACTUAL card before any expensive 27B work.
subprocess.run([sys.executable, str(PROJECT/'scripts/gpu_counter_smoke_v3.py')], check=True)

## Prepare donor + warm state

If local disk is roomy, copying the ~52 GiB donor off Drive makes the 64-block teacher-cache pass much faster.
The ~21 GiB warm state is preferentially extracted into `/dev/shm`.

In [ ]:
# Optional local donor copy; otherwise read shards directly from Drive.
local_free = shutil.disk_usage('/content').free / 2**30
if local_free >= 115:
    LOCAL_DONOR = WORK / 'donor'
    if not (LOCAL_DONOR/'model.safetensors.index.json').exists():
        LOCAL_DONOR.mkdir(parents=True, exist_ok=True)
        subprocess.run(['rsync','-a','--info=progress2',str(DONOR_DIR)+'/',str(LOCAL_DONOR)+'/'], check=True)
else:
    LOCAL_DONOR = DONOR_DIR
print('Teacher donor:', LOCAL_DONOR)

In [ ]:
# Extract warm streamed state and NORMALIZE a possible nested zip root.
shm_free = shutil.disk_usage('/dev/shm').free / 2**30
base = Path('/dev/shm/mn27_state_v3') if shm_free >= 26 else WORK/'mn27_state_v3'
if base.exists(): shutil.rmtree(base)
base.mkdir(parents=True)
with zipfile.ZipFile(WARM_STATE_ZIP) as zf: zf.extractall(base)
manifests = [p for p in base.rglob('manifest.json') if list(p.parent.glob('block_*.pt'))]
assert len(manifests) == 1, f'Expected one streamed-state manifest, found {manifests}'
STATE_DIR = manifests[0].parent
print('STATE_DIR=', STATE_DIR)
print(json.dumps(json.loads((STATE_DIR/'manifest.json').read_text()), indent=2)[:3000])

## Corpus

The old run used a six-domain EN/RU/code/math/science/instruct mix. v3 rebuilds it with the donor tokenizer if a persisted copy is not already on Drive. The train corpus is intentionally modest (5M tokens): the strict run consumes only ~0.41M tokens.

In [ ]:
# Restore/build deterministic six-domain corpus.
if (DATA_DRIVE/'manifest.json').exists():
    if LOCAL_DATA.exists(): shutil.rmtree(LOCAL_DATA)
    LOCAL_DATA.mkdir(parents=True)
    subprocess.run(['rsync','-a',str(DATA_DRIVE)+'/',str(LOCAL_DATA)+'/'], check=True)
else:
    if LOCAL_DATA.exists(): shutil.rmtree(LOCAL_DATA)
    env = os.environ.copy(); env['MODEL'] = str(LOCAL_DONOR)
    subprocess.run([
        sys.executable, str(PROJECT/'scripts/build_mix_corpus.py'),
        '--out', str(LOCAL_DATA), '--train-tokens', '5000000', '--val-tokens', '24000'
    ], env=env, check=True)
    DATA_DRIVE.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync','-a','--info=progress2',str(LOCAL_DATA)+'/',str(DATA_DRIVE)+'/'], check=True)
DATA_DIR = LOCAL_DATA
print((DATA_DIR/'manifest.json').read_text())

In [ ]:
# CPU unit tests for the corrected mathematics/selection/key mapping.
subprocess.run([
    sys.executable, '-m', 'pytest', '-q',
    str(PROJECT/'tests/test_strict_sparse_kd.py'),
    str(PROJECT/'tests/test_strict_gate_v3.py'),
    str(PROJECT/'tests/test_text_only_mapping.py')
], cwd=PROJECT, check=True)

## End-to-end 2-block smoke

This catches exactly the failures that v2 missed: teacher-cache schema, Qwen3.8 text-stack mapping, warm-state remap, counter backward, strict loss and evaluation. It uses only 2 of 64 layers and one tiny step, so it is plumbing validation—not a quality measurement.

In [ ]:
# Build tiny 2-block cache.
SMOKE_CACHE = WORK/'smoke_cache'
SMOKE_OUT = WORK/'smoke_out'
for p in (SMOKE_CACHE, SMOKE_OUT):
    if p.exists(): shutil.rmtree(p)
    p.mkdir(parents=True)
env = os.environ.copy()
env.update({
    'MODEL': str(LOCAL_DONOR), 'DATA_DIR': str(DATA_DIR), 'OUT': str(SMOKE_CACHE),
    'STEPS':'1','BATCH':'1','SEQ':'128','TOPK':'128','SEED':'777',
    'DEVICE':'cuda','DTYPE':'bf16','NUM_BLOCKS':'2','CHUNK_ROWS':'1','SHARD_STEPS':'1','TEMPERATURE':'1.0'
})
subprocess.run([sys.executable, str(PROJECT/'scripts/kd_teacher_cache_v3.py')], env=env, check=True)
subprocess.run([sys.executable, str(PROJECT/'scripts/validate_kd_cache_v3.py'), str(SMOKE_CACHE), '--expected-topk','128','--expected-steps','1'], check=True)

In [ ]:
# One strict student step on the same 2-block teacher cache.
env = os.environ.copy()
env.update({
    'MODEL': str(LOCAL_DONOR), 'STATE_DIR': str(STATE_DIR), 'DATA_DIR': str(DATA_DIR),
    'CACHE': str(SMOKE_CACHE), 'CKPT_DIR': str(SMOKE_OUT), 'DEVICE':'cuda',
    'STEPS':'1','NUM_BLOCKS':'2','KD_T':'1.0','KD_WEIGHT':'1.0','CE_WEIGHT':'1.0',
    'COUNTER_LR_START':'0.000125','COUNTER_LR_END':'0.000025','SCALE_LR':'0.000025',
    'DECIMATION':'1','FP_TRAIN_MODE':'none','GRAD_CKPT':'1','EVAL_EVERY':'1',
    'EVAL_MAX_TOKENS':'1024','EARLY_STOP_PATIENCE':'0','SAVE_BEST':'0',
    'MIN_IMPROVEMENT':'0.002','MAX_DOMAIN_REGRESSION':'0.05'
})
subprocess.run([sys.executable, str(PROJECT/'scripts/kd_cached_strict_v3.py')], env=env, check=True)
print((SMOKE_OUT/'run_summary.json').read_text())
# Free smoke artifacts before the real run.
shutil.rmtree(SMOKE_CACHE, ignore_errors=True); shutil.rmtree(SMOKE_OUT, ignore_errors=True)
torch.cuda.empty_cache()

## Production teacher cache

RTX PRO 6000 96 GB uses K=1024; H100 uses K=512. The cache is around a few GB and is persisted to Drive so an interrupted KD session does not require recomputing the 27B teacher pass. The cache validator compares model/data fingerprints before reuse.

In [ ]:
# Build or restore the exact v3 teacher cache.
TOPK = profile.cache_topk
STEPS = 400
BATCH = profile.batch
SEQ = profile.seq_len

def validate_cache(path):
    if not (path/'cache_manifest.json').exists(): return False
    cmd=[sys.executable, str(PROJECT/'scripts/validate_kd_cache_v3.py'), str(path),
         '--expected-topk',str(TOPK),'--expected-steps',str(STEPS),
         '--model-index',str(LOCAL_DONOR/'model.safetensors.index.json'),
         '--data-manifest',str(DATA_DIR/'manifest.json')]
    return subprocess.run(cmd, check=False).returncode == 0

if CACHE_DRIVE.exists() and validate_cache(CACHE_DRIVE):
    if LOCAL_CACHE.exists(): shutil.rmtree(LOCAL_CACHE)
    LOCAL_CACHE.mkdir(parents=True)
    subprocess.run(['rsync','-a','--info=progress2',str(CACHE_DRIVE)+'/',str(LOCAL_CACHE)+'/'], check=True)
else:
    if LOCAL_CACHE.exists(): shutil.rmtree(LOCAL_CACHE)
    LOCAL_CACHE.mkdir(parents=True)
    env=os.environ.copy(); env.update({
        'MODEL':str(LOCAL_DONOR),'DATA_DIR':str(DATA_DIR),'OUT':str(LOCAL_CACHE),
        'STEPS':str(STEPS),'BATCH':str(BATCH),'SEQ':str(SEQ),'TOPK':str(TOPK),'SEED':'0',
        'DEVICE':'cuda','DTYPE':'bf16','NUM_BLOCKS':'0','CHUNK_ROWS':str(profile.cache_chunk_rows),
        'SHARD_STEPS':'50','TEMPERATURE':'1.0'
    })
    subprocess.run([sys.executable, str(PROJECT/'scripts/kd_teacher_cache_v3.py')], env=env, check=True)
    assert validate_cache(LOCAL_CACHE)
    CACHE_DRIVE.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync','-a','--delete','--info=progress2',str(LOCAL_CACHE)+'/',str(CACHE_DRIVE)+'/'], check=True)
CACHE = LOCAL_CACHE
print(json.dumps(json.loads((CACHE/'cache_manifest.json').read_text()), indent=2))

## Production strict KD

The run is intentionally conservative. **No normal gradient accumulation is used:** counter layers mutate inside every backward, so calling multiple backward passes before an optimizer step would not mean what ordinary `grad_accum` means. v3 freezes all ordinary FP parameters and performs one counter update per batch.

In [ ]:
# Resolve and record the exact production recipe.
recipe = json.loads((PROJECT/'configs/qwen38_27b_strict_kd_v3.json').read_text())
recipe['hardware'] = profile.__dict__
recipe['paths'] = {'donor':str(LOCAL_DONOR),'warm_state_dir':str(STATE_DIR),'data':str(DATA_DIR),'cache':str(CACHE),'output':str(LOCAL_OUTPUT)}
(LOCAL_OUTPUT/'resolved_recipe_v3.json').write_text(json.dumps(recipe, indent=2))
print(json.dumps(recipe, indent=2))

In [ ]:
# Run strict KD. Warm step 0 stays authoritative unless a KD eval passes every gate.
# Remove stale v3 training outputs but keep preflight/recipe.
for name in ['metrics.json','selection.json','best.pt','selected_artifact.json','run_summary.json','KD_ACCEPTED.txt','USE_WARM_STATE.txt','restore_meta.json']:
    p=LOCAL_OUTPUT/name
    if p.exists(): p.unlink()

env=os.environ.copy(); env.update({
    'MODEL':str(LOCAL_DONOR),'STATE_DIR':str(STATE_DIR),'DATA_DIR':str(DATA_DIR),
    'CACHE':str(CACHE),'CKPT_DIR':str(LOCAL_OUTPUT),'CKPT_TMP':str(TMP_CKPT),'DEVICE':'cuda',
    'STEPS':'400','KD_T':'1.0','KD_WEIGHT':'1.0','CE_WEIGHT':'1.0',
    'COUNTER_LR_START':'0.000125','COUNTER_LR_END':'0.000025','SCALE_LR':'0.000025',
    'FP_LR':'0.00002','GRAD_CLIP':'0.5','STATS_SCOPE':'group','DECIMATION':'1',
    'EVAL_EVERY':'25','EVAL_MAX_TOKENS':'24000','NUM_BLOCKS':'0','LOG_EVERY':'10',
    'GRAD_CKPT':'1','FP_TRAIN_MODE':'none','SAVE_BEST':'1','EARLY_STOP_PATIENCE':'8',
    'MIN_IMPROVEMENT':'0.002','MAX_DOMAIN_REGRESSION':'0.05'
})
logfile=LOCAL_OUTPUT/'strict_v3_console.log'
with logfile.open('w', buffering=1) as log:
    proc=subprocess.Popen([sys.executable, str(PROJECT/'scripts/kd_cached_strict_v3.py')], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end=''); log.write(line)
    rc=proc.wait()
if rc != 0:
    print(logfile.read_text(errors='replace')[-16000:])
    raise RuntimeError(f'strict v3 failed with rc={rc}')

In [ ]:
# Authoritative selection.
selection=json.loads((LOCAL_OUTPUT/'selection.json').read_text())
print(json.dumps(selection, indent=2, ensure_ascii=False))
if selection['accepted_kd']:
    assert (LOCAL_OUTPUT/'best.pt').exists(), 'Gate accepted KD but deployable best.pt is missing'
else:
    assert (LOCAL_OUTPUT/'USE_WARM_STATE.txt').exists()
print((LOCAL_OUTPUT/'run_summary.json').read_text())

In [ ]:
# Sync metadata, logs, and ONLY a gate-approved KD artifact to Drive.
OUTPUT_DRIVE.mkdir(parents=True, exist_ok=True)
small = ['preflight_v3.json','resolved_recipe_v3.json','metrics.json','selection.json','selected_artifact.json','run_summary.json','strict_v3_console.log','restore_meta.json','KD_ACCEPTED.txt','USE_WARM_STATE.txt']
for name in small:
    p=LOCAL_OUTPUT/name
    if p.exists(): shutil.copy2(p, OUTPUT_DRIVE/name)
if selection['accepted_kd']:
    subprocess.run(['rsync','-a','--info=progress2',str(LOCAL_OUTPUT/'best.pt'),str(OUTPUT_DRIVE/'best.pt')], check=True)
(OUTPUT_DRIVE/'SOURCE_POINTERS.json').write_text(json.dumps({
    'release_zip':str(RELEASE_ZIP),'donor':str(DONOR_DIR),'warm_state_zip':str(WARM_STATE_ZIP),
    'data':str(DATA_DRIVE),'teacher_cache':str(CACHE_DRIVE)
}, indent=2))
(OUTPUT_DRIVE/'_COMPLETE').write_text('strict-v3 run finished and selection.json is authoritative\n')
print('Synced:', OUTPUT_DRIVE)

## Interpretation

- `KD_ACCEPTED.txt` + `best.pt`: a strict-alpha=0 KD candidate beat warm aggregate metric by the configured margin **and** stayed within the 5% regression cap in every domain.
- `USE_WARM_STATE.txt`: KD did not earn deployment; keep `/MyDrive/mn27/state_max.zip`.
- `selection.json` is authoritative.
- A falling KD loss alone is **not** evidence of improvement; strict per-domain PPL is the decision signal.